In [61]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import (
    StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
)
from pyspark.ml import Pipeline
from pyspark.sql.functions import col, when, isnull

# %%
# Configurar SparkSession
spark = SparkSession.builder \
    .appName("SECOP_FeatureEngineering") \
    .master("spark://spark-master:7077") \
    .config("spark.executor.memory", "2g") \
    .getOrCreate()

print(f"Spark Version: {spark.version}")
print(f"Spark Master: {spark.sparkContext.master}")

Spark Version: 3.5.0
Spark Master: spark://spark-master:7077


In [62]:
# Cargar datos
df = spark.read.parquet("/opt/spark-data/processed/secop_eda.parquet")
print(f"Registros cargados: {df.count():,}")

Registros cargados: 100,000


In [63]:
# Explorar columnas disponibles
print("Columnas disponibles:")
for col_name in df.columns:
    print(f"  - {col_name}")

Columnas disponibles:
  - referencia_del_contrato
  - nit_entidad
  - nombre_entidad
  - departamento
  - ciudad
  - tipo_de_contrato
  - valor_del_contrato
  - fecha_de_firma
  - duraci_n_del_contrato
  - proveedor_adjudicado
  - estado_contrato
  - year
  - month
  - valor_del_contrato_num
  - fecha_de_firma_parsed
  - anio
  - mes


In [64]:
# ## RETO 1: Selección de Features
label_col = "valor_del_contrato_num"
feature_num_cols = ["month"]   # <- solo features

# 2) Variables categóricas seleccionadas 
categorical_cols = ["departamento", "tipo_de_contrato", "estado_contrato"]

# 3) Variables numéricas seleccionadas 
numeric_cols = ["month", "valor_del_contrato_num"] 

# Verificar qué columnas existen realmente
available_cat = [c for c in categorical_cols if c in df.columns]
available_num = [c for c in feature_num_cols if c in df.columns]

print(f"Categóricas seleccionadas: {available_cat}")
print(f"Numéricas seleccionadas: {available_num}")
print(f"Label: {label_col}")
# 4) Justificación de la selección:
# Objetivo (label): valor_del_contrato_num
# - Se usa como variable objetivo porque representa el valor del contrato en formato numérico,
#   requerido para un modelo de regresión en Spark ML
#
# Variables categóricas seleccionadas:
# - departamento: Recoge la heterogeneidad regional. Factores como la ubicación geográfica, costos logísticos y la oferta local de proveedores son determinantes en la formación de precios
# - tipo_de_contrato: Define la arquitectura de costos base. La naturaleza de un contrato (obra civil frente a prestación de servicios o suministros) dicta márgenes y presupuestos radicalmente distintos
# - estado_contrato: es un indicador de la complejidad administrativa y técnica
#
# Variables numéricas seleccionadas:
# - month: Captura la estacionalidad del gasto público. Permite identificar ciclos de ejecución presupuestal, como el aumento de contrataciones hacia el cierre del año fiscal

Categóricas seleccionadas: ['departamento', 'tipo_de_contrato', 'estado_contrato']
Numéricas seleccionadas: ['month']
Label: valor_del_contrato_num


In [70]:
#RETO 2
# Estrategia elegida: Opción A (dropna)
# Justificación:
# - Para un pipeline de ML es clave no tener nulos en features ni en la variable objetivo
# - ya que evita introducir sesgo por imputación y deja un dataset consistente para StringIndexer/OneHotEncoder y el modelo
# - Dado que year/month/label son críticos, preferimos eliminar registros incompletos para mantener calidad.

label_col = "valor_del_contrato_num"

# Asegurar tipos numéricos 
df_cast = df \
    .withColumn("month", col("month").cast("int")) \
    .withColumn(label_col, col(label_col).cast("double"))

df_clean = df_cast.dropna(subset=available_cat + available_num + [label_col])

print(f"Registros después de limpiar: {df_clean.count():,}")


Registros después de limpiar: 100,000


In [66]:
# Convierte strings a índices numéricos
indexers = [
    StringIndexer(inputCol=col, outputCol=col + "_idx", handleInvalid="keep")
    for col in available_cat
]

In [67]:
# ## PASO 1: StringIndexer para Variables Categóricas
from pyspark.ml.feature import StringIndexer

indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
    for c in available_cat
]

print("StringIndexers:")
for idx in indexers:
    print(" -", idx.getInputCol(), "->", idx.getOutputCol())


StringIndexers:
 - departamento -> departamento_idx
 - tipo_de_contrato -> tipo_de_contrato_idx
 - estado_contrato -> estado_contrato_idx


In [68]:
# PASO 2: OneHotEncoder para generar variables dummy
from pyspark.ml.feature import OneHotEncoder

encoders = [
    OneHotEncoder(inputCol=f"{c}_idx", outputCol=f"{c}_vec")
    for c in available_cat
]

print("OneHotEncoders:")
for enc in encoders:
    print(" -", enc.getInputCol(), "->", enc.getOutputCol())

OneHotEncoders:
 - departamento_idx -> departamento_vec
 - tipo_de_contrato_idx -> tipo_de_contrato_vec
 - estado_contrato_idx -> estado_contrato_vec


In [71]:
# RETO 3: VectorAssembler para combinar todas las features
# Combinamos: features numéricas + features categóricas codificadas
feature_cols = []
feature_cols += available_num
feature_cols += [f"{c}_vec" for c in available_cat]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features_raw")


print("VectorAssembler inputCols:", feature_cols)

VectorAssembler inputCols: ['month', 'departamento_vec', 'tipo_de_contrato_vec', 'estado_contrato_vec']


In [72]:
# RETO 4: Construir Pipeline
# Pipeline = secuencia de transformaciones
pipeline_stages = []
pipeline_stages += indexers
pipeline_stages += encoders
pipeline_stages += [assembler]

pipeline = Pipeline(stages=pipeline_stages)

print("Pipeline stages:", [type(s).__name__ for s in pipeline_stages])

Pipeline stages: ['StringIndexer', 'StringIndexer', 'StringIndexer', 'OneHotEncoder', 'OneHotEncoder', 'OneHotEncoder', 'VectorAssembler']


In [73]:
# PASO 5: Entrenar el pipeline (fit)
# Nota: StringIndexer y OneHotEncoder necesitan "aprender" del dataset
print("Entrenando pipeline...")
pipeline_model = pipeline.fit(df_clean)
print("✓ Pipeline entrenado")

df_transformed = pipeline_model.transform(df_clean)

# Spark ML espera columna 'label'
df_model = df_transformed.withColumnRenamed(label_col, "label")

df_model.select("label", "features_raw").show(5, truncate=False)

sample_features = df_model.select("features_raw").first()[0]
print("Dimensión final features_raw:", len(sample_features))
print("Registros finales:", df_model.count())

Entrenando pipeline...


✓ Pipeline entrenado
+------------+-----------------------------------+
|label       |features_raw                       |
+------------+-----------------------------------+
|2.067E7     |(58,[0,14,35,52],[1.0,1.0,1.0,1.0])|
|3.0848588E7 |(58,[0,2,35,51],[1.0,1.0,1.0,1.0]) |
|9.5E7       |(58,[0,1,35,52],[1.0,1.0,1.0,1.0]) |
|1.06659819E8|(58,[0,1,35,51],[1.0,1.0,1.0,1.0]) |
|7200000.0   |(58,[0,8,35,51],[1.0,1.0,1.0,1.0]) |
+------------+-----------------------------------+
only showing top 5 rows

Dimensión final features_raw: 58
Registros finales: 100000


In [74]:
# %%
# Verificar el resultado
print("\nEsquema de features_raw:")
df_transformed.select("features_raw").printSchema()


Esquema de features_raw:
root
 |-- features_raw: vector (nullable = true)



In [75]:
# Ver dimensión del vector de features
sample_features = df_transformed.select("features_raw").first()[0]
print(f"Dimensión del vector de features: {len(sample_features)}")

Dimensión del vector de features: 58


In [76]:
# Ver dimensión del vector de features
sample_features = df_transformed.select("features_raw").first()[0]
print(f"Dimensión del vector de features: {len(sample_features)}")

# %%
# Mostrar ejemplo de transformación
df_transformed.select(
    available_cat[0] if available_cat else "id",
    available_cat[0] + "_idx" if available_cat else "id",
    available_cat[0] + "_vec" if available_cat else "id",
    "features_raw"
).show(5, truncate=True)

Dimensión del vector de features: 58
+--------------------+----------------+----------------+--------------------+
|        departamento|departamento_idx|departamento_vec|        features_raw|
+--------------------+----------------+----------------+--------------------+
|            Casanare|            13.0| (34,[13],[1.0])|(58,[0,14,35,52],...|
|           Antioquia|             1.0|  (34,[1],[1.0])|(58,[0,2,35,51],[...|
|Distrito Capital ...|             0.0|  (34,[0],[1.0])|(58,[0,1,35,52],[...|
|Distrito Capital ...|             0.0|  (34,[0],[1.0])|(58,[0,1,35,51],[...|
|           Magdalena|             7.0|  (34,[7],[1.0])|(58,[0,8,35,51],[...|
+--------------------+----------------+----------------+--------------------+
only showing top 5 rows



In [78]:
# %%
# Guardar pipeline entrenado
pipeline_path = "/opt/spark-data/processed/feature_pipeline"
pipeline_model.write().overwrite().save(pipeline_path)
print(f"✓ Pipeline guardado en: {pipeline_path}")

# %%
# Guardar dataset transformado
output_path = "/opt/spark-data/processed/secop_features.parquet"
df_model.write.mode("overwrite").parquet(output_path)
print(f"Dataset transformado guardado en: {output_path}")

✓ Pipeline guardado en: /opt/spark-data/processed/feature_pipeline


Dataset transformado guardado en: /opt/spark-data/processed/secop_features.parquet


In [57]:
# 1. Pipeline:
# Usamos Pipeline porque garantiza un flujo reproducible y ordenado de transformaciones (fit/transform) en un solo objeto
# Esto evita inconsistencias entre train y test, facilita mantenimiento, validación cruzada y despliegue (misma lógica siempre)

# 2. Orden de transformaciones:
# OneHotEncoder necesita entradas numéricas (índices). Si se aplica antes de StringIndexer, fallará porque no puede codificar
# directamente strings/categorías; además no existirían las columnas *_idx que usa como input.

# 3. StandardScaler:
# Se usa cuando los modelos son sensibles a la escala de las variables numéricas (p. ej., regresión con regularización L1/L2,
# modelos lineales, SVM, K-means). Ayuda a que una variable con magnitud grande no domine el entrenamiento.

# 4. Guardar pipeline:
# Guardar pipeline_model permite reutilizar exactamente las mismas transformaciones aprendidas (mapeo de categorías, OHE, etc.)
# para nuevos datos en producción. Evita recalcular/alterar índices de categorías y asegura consistencia en inferencia.


In [79]:
# %%
print("\n" + "="*60)
print("RESUMEN FEATURE ENGINEERING")
print("="*60)
print(f"✓ Variables categóricas procesadas: {len(available_cat)}")
print(f"✓ Variables numéricas: {len(available_num)}")
print(f"✓ Dimensión final del vector: {len(sample_features)}")
print(f"✓ Pipeline guardado y listo para usar")
print("="*60)

# %%
spark.stop()


RESUMEN FEATURE ENGINEERING
✓ Variables categóricas procesadas: 3
✓ Variables numéricas: 1
✓ Dimensión final del vector: 58
✓ Pipeline guardado y listo para usar
